# Estimator Sheet 
This notebook builds the single combined dataset used for quantile regression estimation. It walks a folder of forecast-card CSVs (`project-*.csv`, `poi-*.csv`, `observations-*.csv`), finds every project, combines each project's files into one merged table, adds K/D factors, county-level socioeconomic data, peak-hour volume, and post-construction capacity, then writes everything out to a single `combined_output.csv`. That combined file is the input for the model-fitting notebook.

**How to use this notebook:**
1. Run the "find project folders" cell to confirm every project was located. Set `DATA_ROOT` to the folder containing all your project CSVs.
2. Run the "build the combined dataset" cell. For each project this loads its project, poi, and observations CSVs, merges them together, and labels each observation as `before` or `after` the project's actual open date. All projects are then stacked into one `master_df`.
3. Run the "years from open" cell to add a `years_from_open` column to `master_df`, the difference between each observation's year and the project's actual open year.
4. Run the "K/D factors" cell to add `k_factor` and `d_factor` to `master_df`.
5. Run the "add county-level socioeconomic data" cell to merge in population, employment, pay, and fuel-price columns from `us_county_socioeconomic_2000_2025.csv` (matched on state, county, and year). These columns land right after `d_factor` and are also what the capacity equation's population/income growth terms pull from.
6. Run the "peak-hour volume" cell to add `peak_hour_volume`. It runs after the socioeconomic merge so that, in `combined_output.csv`, the socioeconomic columns end up sitting between `d_factor` and `peak_hour_volume`.
7. Run the "post-construction capacity" cells to add `engineering_capacity_ceng`, `capacity_multiplier`, and `peak_hour_capacity` to `master_df`, via the **Post-Construction Capacity Expansion Equation**.
8. Run the "spot-check" cell to visually confirm the merges and factor calculations worked as expected.
9. Run the "write out" cell to add a blank `comment` column and save `master_df` to `combined_output.csv`.

In [1]:
import pandas as pd
import numpy as np
import glob
import os
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

## 1. Find your project folders

Point `DATA_ROOT` at the folder holding your project data. This walks the entire directory tree underneath it and collects every sub-folder that contains a `project-*.csv` file (each matching folder is treated as one project). The result is a sorted list of folder paths, printed below so you can confirm the right projects were found.

**Update this path to point at your own data folder before continuing.**

In [ ]:
def find_project_folders(root_dir):
    """Return every sub-folder of root_dir that contains a project-*.csv file."""
    folders = []
    for dirpath, _dirnames, filenames in os.walk(root_dir):
        if any(re.match(r"project-.*\.csv$", f, re.IGNORECASE) for f in filenames):
            folders.append(dirpath)
    return sorted(folders)

DATA_ROOT = r"D:\Documents\Code\forecastcards_ba_data\data"
project_folders = find_project_folders(DATA_ROOT)
project_folders

['C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\KY_1-180.10',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\KY_1-180.11',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\KY_1-180.51',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\KY_1-314.10',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\KY_1-8002.00',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\KY_10-138.00_NI',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\KY_10-166.00',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\KY_10-280.00',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\KY_11-11.00',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\KY_11-188.00',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\KY_11-20.00_IR',
 'C:\\Users\\User\\Documents\\Code\\forecastcards_ba_data\\data\\KY_11-8509.00_NR',
 'C:\\Users\\User\\Documents\\Code\\

## 2. Build the combined dataset

This section does the actual finding, loading, and merging for every project, then stacks the results into one master table used for estimation.

**Loading a project's files** — for a given project folder, finds and reads:
- `project-*.csv` → project-level metadata (one row per project)
- `poi-*.csv` → "points of interest" tied to that project (e.g. intersections/segments)
- `observations-*.csv` → observed traffic/volume records

**Merging a project's files** — performs two sequential merges:
- **project → poi:** tags each POI row with its `project_id`, then merges project metadata onto every POI row (keeping all POIs, broadcasting the single project row across them).
- **before/after labeling:** converts `observation_date` to a real datetime and compares it to the project's `date_open_actual` to create `before_or_after`.
- **(project+poi) → observations:** merges the project/poi table onto the observations, matching on `poi_id` **and** `before_or_after`, keeping every observation row, and renames `start_time`/`end_time` to the `_observation` versions for consistent naming.


In [3]:
def load_project_files(folder):
    """Load the project / poi / observations csvs for one folder."""
    project_path = glob.glob(os.path.join(folder, "project-*.csv"))[0]
    poi_path = glob.glob(os.path.join(folder, "poi-*.csv"))[0]
    obs_path = glob.glob(os.path.join(folder, "observations-*.csv"))[0]

    project_df = pd.read_csv(project_path)
    poi_df = pd.read_csv(poi_path)
    obs_df = pd.read_csv(obs_path)

    return project_df, poi_df, obs_df

In [4]:
def process_project_folder(folder):
    project_df, poi_df, obs_df = load_project_files(folder)

    project_id = project_df["project_id"].iloc[0]
    date_open_actual = pd.to_datetime(project_df["date_open_actual"].iloc[0])
    date_open_planned = pd.to_datetime(project_df["date_open_planned"].iloc[0])

    # use the actual open date when available; otherwise fall back to the planned open date
    date_open_effective = date_open_actual if pd.notna(date_open_actual) else date_open_planned

    poi_df = poi_df.copy()

    if "before_or_after" in poi_df.columns:
        poi_df["before_or_after"] = (
            poi_df["before_or_after"].astype("string").str.strip().str.lower()
        )
    poi_df["project_id"] = project_id

    # STEP 1: project -> poi
    project_poi = pd.merge(project_df, poi_df, on="project_id", how="right")

    # derive a sensible before/after label for each observation
    obs_df = obs_df.copy()
    obs_df["observation_date"] = pd.to_datetime(obs_df["observation_date"], errors="coerce")
    obs_df["before_or_after"] = pd.array(
        np.where(obs_df["observation_date"] < date_open_effective, "before", "after"),
        dtype="string",
    )
    obs_df["before_or_after_binary"] = (obs_df["before_or_after"] == "after").astype(int)

    # STEP 2: (project + poi) -> observation
    merged = pd.merge(
        project_poi, obs_df, on=["poi_id", "before_or_after"], how="right"
    )

    merged = merged.rename(columns={"start_time": "start_time_observation",
                                     "end_time": "end_time_observation"})

    merged["date_open_effective"] = date_open_effective

    return merged

In [5]:
project_tables = [process_project_folder(folder) for folder in project_folders]

master_df = pd.concat(project_tables, ignore_index=True, sort=False)

print(f"{len(master_df)} total observation rows across {len(project_tables)} project(s)")
master_df.head(10)

4841 total observation rows across 153 project(s)


,project_id,state,county,project_type,description,date_open_planned,date_horizon,date_open_actual,area_type,project_length_miles,poi_id,before_or_after,facility_name,segment_length_miles,speed_limit_mph,lane_width,signals_per_mile,stop_signs_per_mile,through_lanes,left_turn_lanes,right_turn_lanes,express_hov_hot_lanes,shoulder_width,functional_class,obs_id,observation_date,start_time_observation,end_time_observation,observation_variable,observation_value,observation_unit,before_or_after_binary,date_open_effective,construction_start_date
0,1-180.10,KY,Trigg,Major Widen,Widen US-68 from MP 11.382 - 12.740,1/1/2020,NaN,NaN,Rural Area,1.358,111542,before,CANTON RD,NaN,55,11,unknown,unknown,2,1,1,unknown,unknown,Principal Arterial - Other,111542_1,2001-01-01,0:00:00,24:00:00,volume,3430,AADT,0,2020-01-01,NaN
1,1-180.10,KY,Trigg,Major Widen,Widen US-68 from MP 11.382 - 12.740,1/1/2020,NaN,NaN,Rural Area,1.358,111542,before,CANTON RD,NaN,55,11,unknown,unknown,2,1,1,unknown,unknown,Principal Arterial - Other,111542_2,2004-01-01,0:00:00,24:00:00,volume,3780,AADT,0,2020-01-01,NaN
2,1-180.10,KY,Trigg,Major Widen,Widen US-68 from MP 11.382 - 12.740,1/1/2020,NaN,NaN,Rural Area,1.358,111542,before,CANTON RD,NaN,55,11,unknown,unknown,2,1,1,unknown,unknown,Principal Arterial - Other,111542_3,2007-01-01,0:00:00,24:00:00,volume,3620,AADT,0,2020-01-01,NaN
3,1-180.10,KY,Trigg,Major Widen,Widen US-68 from MP 11.382 - 12.740,1/1/2020,NaN,NaN,Rural Area,1.358,111542,before,CANTON RD,NaN,55,11,unknown,unknown,2,1,1,unknown,unknown,Principal Arterial - Other,111542_4,2010-01-01,0:00:00,24:00:00,volume,3360,AADT,0,2020-01-01,NaN
4,1-180.10,KY,Trigg,Major Widen,Widen US-68 from MP 11.382 - 12.740,1/1/2020,NaN,NaN,Rural Area,1.358,111542,before,CANTON RD,NaN,55,11,unknown,unknown,2,1,1,unknown,unknown,Principal Arterial - Other,111542_5,2013-01-01,0:00:00,24:00:00,volume,3914,AADT,0,2020-01-01,NaN
5,1-180.10,KY,Trigg,Major Widen,Widen US-68 from MP 11.382 - 12.740,1/1/2020,NaN,NaN,Rural Area,1.358,111542,before,CANTON RD,NaN,55,11,unknown,unknown,2,1,1,unknown,unknown,Principal Arterial - Other,111542_6,2016-01-01,0:00:00,24:00:00,volume,4494,AADT,0,2020-01-01,NaN
6,1-180.10,KY,Trigg,Major Widen,Widen US-68 from MP 11.382 - 12.740,1/1/2020,NaN,NaN,Rural Area,1.358,111542,before,CANTON RD,NaN,55,11,unknown,unknown,2,1,1,unknown,unknown,Principal Arterial - Other,111542_7,2019-01-01,0:00:00,24:00:00,volume,5307,AADT,0,2020-01-01,NaN
7,1-180.10,KY,Trigg,Major Widen,Widen US-68 from MP 11.382 - 12.740,1/1/2020,NaN,NaN,Rural Area,1.358,111542,after,CANTON RD,NaN,55,12,unknown,unknown,4,1,1,unknown,unknown,Principal Arterial - Other,111542_8,2021-01-01,0:00:00,24:00:00,volume,5147,AADT,1,2020-01-01,NaN
8,1-180.10,KY,Trigg,Major Widen,Widen US-68 from MP 11.382 - 12.740,1/1/2020,NaN,NaN,Rural Area,1.358,111542,after,CANTON RD,NaN,55,12,unknown,unknown,4,1,1,unknown,unknown,Principal Arterial - Other,111542_9,2022-01-01,0:00:00,24:00:00,volume,4580,AADT,1,2020-01-01,NaN
9,1-180.10,KY,Trigg,Major Widen,Widen US-68 from MP 11.382 - 12.740,1/1/2020,NaN,NaN,Rural Area,1.358,111542,after,CANTON RD,NaN,55,12,unknown,unknown,4,1,1,unknown,unknown,Principal Arterial - Other,111542_10,2025-01-01,0:00:00,24:00:00,volume,5342,AADT,1,2020-01-01,NaN


## 3. Years from open (before/after completion year)

Adds a `years_from_open` column to `master_df`: the difference between the calendar year of each observation and the calendar year the project actually opened (`date_open_actual`).

In [6]:
master_df["observation_date"] = pd.to_datetime(master_df["observation_date"], errors="coerce")
master_df["date_open_actual"] = pd.to_datetime(master_df["date_open_actual"], errors="coerce")
master_df["date_open_planned"] = pd.to_datetime(master_df["date_open_planned"], errors="coerce")

# use the actual open date when available; otherwise fall back to the planned open date
master_df["date_open_effective"] = master_df["date_open_actual"].fillna(master_df["date_open_planned"])

master_df["years_from_open"] = (
    master_df["observation_date"].dt.year - master_df["date_open_effective"].dt.year
)

master_df[["project_id", "poi_id", "observation_date", "date_open_actual", "date_open_planned", "date_open_effective", "years_from_open"]].head(10)

,project_id,poi_id,observation_date,date_open_actual,date_open_planned,date_open_effective,years_from_open
0,1-180.10,111542,2001-01-01,NaT,2020-01-01,2020-01-01,-19.0
1,1-180.10,111542,2004-01-01,NaT,2020-01-01,2020-01-01,-16.0
2,1-180.10,111542,2007-01-01,NaT,2020-01-01,2020-01-01,-13.0
3,1-180.10,111542,2010-01-01,NaT,2020-01-01,2020-01-01,-10.0
4,1-180.10,111542,2013-01-01,NaT,2020-01-01,2020-01-01,-7.0
5,1-180.10,111542,2016-01-01,NaT,2020-01-01,2020-01-01,-4.0
6,1-180.10,111542,2019-01-01,NaT,2020-01-01,2020-01-01,-1.0
7,1-180.10,111542,2021-01-01,NaT,2020-01-01,2020-01-01,1.0
8,1-180.10,111542,2022-01-01,NaT,2020-01-01,2020-01-01,2.0
9,1-180.10,111542,2025-01-01,NaT,2020-01-01,2020-01-01,5.0


## 4. K/D factors

Adds `k_factor` and `d_factor` using `K_D_Factors.csv`, matched to each row's `functional_class`:

- `k_factor`, `d_factor` — looked up directly from `K_D_Factors.csv`.

Rows with `functional_class == "unknown"` will come out blank for these columns.

In [ ]:
# Paths to the factor tables
FACTOR_TABLES_ROOT = r"D:\Documents\Code\forecastcards_v2\forecastcards\capacity_factor_tables"

K_D_FACTORS_PATH = os.path.join(FACTOR_TABLES_ROOT, "K_D_Factors.csv")
CAP_CAL_PATH = os.path.join(FACTOR_TABLES_ROOT, "Cap_Cal_Values.csv")

def normalize_functional_class(value):
    # Lowercase/trim a functional class label so it matches across files
    # (e.g. K_D_Factors.csv uses 'Local', Cap_Base_Values.csv uses 'Local Road').
    if pd.isna(value):
        return None
    label = str(value).strip().lower()
    if label == "local road":
        label = "local"
    return label

# K / D factors by functional class
kd_factors = pd.read_csv(K_D_FACTORS_PATH)[["Functional Class", "K Factor", "D Factor"]].dropna(subset=["Functional Class"])
kd_factors["_fc"] = kd_factors["Functional Class"].apply(normalize_functional_class)
k_factor_lookup = kd_factors.set_index("_fc")["K Factor"].to_dict()
d_factor_lookup = kd_factors.set_index("_fc")["D Factor"].to_dict()

master_df["through_lanes"] = pd.to_numeric(master_df["through_lanes"], errors="coerce")

_fc_norm = master_df["functional_class"].apply(normalize_functional_class)
master_df["k_factor"] = _fc_norm.map(k_factor_lookup)
master_df["d_factor"] = _fc_norm.map(d_factor_lookup)

master_df[["functional_class", "through_lanes", "k_factor", "d_factor"]].head(10)

,functional_class,through_lanes,k_factor,d_factor
0,Principal Arterial - Other,2.0,NaN,NaN
1,Principal Arterial - Other,2.0,NaN,NaN
2,Principal Arterial - Other,2.0,NaN,NaN
3,Principal Arterial - Other,2.0,NaN,NaN
4,Principal Arterial - Other,2.0,NaN,NaN
5,Principal Arterial - Other,2.0,NaN,NaN
6,Principal Arterial - Other,2.0,NaN,NaN
7,Principal Arterial - Other,4.0,NaN,NaN
8,Principal Arterial - Other,4.0,NaN,NaN
9,Principal Arterial - Other,4.0,NaN,NaN


## 5. Add county-level socioeconomic data

Merges in columns from `us_county_socioeconomic_2000_2025.csv` (population, employment, avg annual pay, fuel prices, and their 2025-dollar equivalents) onto `master_df`, matched on **state + county + year**:

- `state` (abbreviation, e.g. `MN`) is mapped to the socioeconomic file's full state name.
- `county` is normalized by stripping the 'County' suffixes so `"Ramsey"` matches `"Ramsey County"`.
- Combined-county POIs (e.g. `"Hennepin and Ramsey"`) are matched by averaging the socioeconomic values across each constituent county for that year.
- The year used for matching is the calendar year of `observation_date`.
- The socioeconomic file's own `State`, `County`, and `Year` columns are **not** included in the output.
- Rows with no match (e.g. observation years outside 2000–2025) come out blank for these columns.

This runs right after `k_factor`/`d_factor` are added and before `peak_hour_volume`, so these columns land between `d_factor` and `peak_hour_volume` in the final `combined_output.csv`. It also has to run before the Post-Construction Capacity Expansion Equation below, since that equation's `Δln(Pop_r)` and `Δln(Inc_r)` terms are built from the `population` and `avg annual pay (2025$)` columns added here.

In [8]:
US_STATE_ABBREV = {
    "Alabama": "AL", "Alaska": "AK", "Arizona": "AZ", "Arkansas": "AR", "California": "CA",
    "Colorado": "CO", "Connecticut": "CT", "Delaware": "DE", "Florida": "FL", "Georgia": "GA",
    "Hawaii": "HI", "Idaho": "ID", "Illinois": "IL", "Indiana": "IN", "Iowa": "IA",
    "Kansas": "KS", "Kentucky": "KY", "Louisiana": "LA", "Maine": "ME", "Maryland": "MD",
    "Massachusetts": "MA", "Michigan": "MI", "Minnesota": "MN", "Mississippi": "MS", "Missouri": "MO",
    "Montana": "MT", "Nebraska": "NE", "Nevada": "NV", "New Hampshire": "NH", "New Jersey": "NJ",
    "New Mexico": "NM", "New York": "NY", "North Carolina": "NC", "North Dakota": "ND", "Ohio": "OH",
    "Oklahoma": "OK", "Oregon": "OR", "Pennsylvania": "PA", "Rhode Island": "RI", "South Carolina": "SC",
    "South Dakota": "SD", "Tennessee": "TN", "Texas": "TX", "Utah": "UT", "Vermont": "VT",
    "Virginia": "VA", "Washington": "WA", "West Virginia": "WV", "Wisconsin": "WI", "Wyoming": "WY",
    "District of Columbia": "DC",
}

COUNTY_SUFFIXES = [
    " County", " Parish", " Borough", " Census Area", " Municipality",
    " Municipio", " City and Borough", " city", " City", " Region",
]

def normalize_county(name):
    # Strip common county/parish/borough suffixes so names match across files.
    if pd.isna(name):
        return None
    name = str(name).strip()
    for suf in COUNTY_SUFFIXES:
        if name.endswith(suf):
            return name[: -len(suf)].strip()
    return name

SOCIO_PATH = os.path.join(FACTOR_TABLES_ROOT, "us_county_socioeconomic_2000_2025.csv")

socio_df = pd.read_csv(SOCIO_PATH)
socio_df["state_abbrev"] = socio_df["State"].map(US_STATE_ABBREV)
socio_df["county_clean"] = socio_df["County"].apply(normalize_county)

SOCIO_VALUE_COLS = [
    c for c in socio_df.columns
    if c not in ("State", "County", "Year", "state_abbrev", "county_clean")
]

socio_lookup = socio_df.groupby(["state_abbrev", "county_clean", "Year"])[SOCIO_VALUE_COLS].mean()

master_df["observation_year"] = pd.to_datetime(master_df["observation_date"], errors="coerce").dt.year

def lookup_socio(row):
    key = (row["state"], row["county"], row["observation_year"])
    if key in socio_lookup.index:
        return socio_lookup.loc[key]
    return pd.Series({c: np.nan for c in SOCIO_VALUE_COLS})

socio_matched = master_df.apply(lookup_socio, axis=1)
socio_matched.columns = [c.lower() for c in socio_matched.columns]
n_unmatched = socio_matched.isna().all(axis=1).sum()
if n_unmatched:
    print(f"WARNING: {n_unmatched} row(s) had no socioeconomic match (check state/county/year):")
    print(master_df.loc[socio_matched.isna().all(axis=1), ["state", "county", "observation_year"]].drop_duplicates())

master_df = pd.concat([master_df, socio_matched], axis=1).drop(columns=["observation_year"])

master_df[["state", "county", "d_factor"] + list(socio_matched.columns)].head(10)

     state  county  observation_year
10      KY   Trigg              1996
36      KY   Trigg              1998
79      KY  Powell              1996
80      KY  Powell              1997
81      KY  Powell              1998
...    ...     ...               ...
2719    KY    Boyd              1996
2720    KY    Boyd              1998
2721    KY    Boyd              1999
2744    KY    Boyd              1997
4136    MN  Ramsey              1997

[126 rows x 3 columns]


,state,county,d_factor,population,employment,avg annual pay,gasoline price,diesel price,cpi adjustment factor (to 2025$),avg annual pay (2025$),gasoline price (2025$),diesel price (2025$)
0,KY,Trigg,NaN,12803.0,3520.0,23497.0,1.472000,1.279747,1.817860,42714.26,2.68,2.33
1,KY,Trigg,NaN,13438.0,3910.0,29360.0,1.897333,1.648186,1.704304,50038.36,3.23,2.81
2,KY,Trigg,NaN,14079.0,4057.0,30938.0,2.698772,2.617731,1.552715,48037.89,4.19,4.06
3,KY,Trigg,NaN,14305.0,3247.0,28043.0,2.745000,2.670000,1.476423,41403.34,4.05,3.94
4,KY,Trigg,NaN,14085.0,3367.0,30255.0,3.495000,3.535000,1.381985,41811.95,4.83,4.89
5,KY,Trigg,NaN,13884.0,3050.0,32144.0,2.130000,2.072500,1.341390,43117.64,2.86,2.78
6,KY,Trigg,NaN,14037.0,3077.0,33383.0,2.617500,2.712500,1.259277,42038.45,3.30,3.42
7,KY,Trigg,NaN,14219.0,2912.0,35358.0,2.870000,2.780000,1.188113,42009.30,3.41,3.30
8,KY,Trigg,NaN,14331.0,2831.0,37282.0,4.040000,4.335000,1.100077,41013.07,4.44,4.77
9,KY,Trigg,NaN,14512.0,2954.0,41639.0,3.155000,3.267500,1.000000,41639.00,3.16,3.27


## 6. Peak-hour volume

Adds `peak_hour_volume` = `AADT x K x D`, computed from the observed volume column (only where the unit is `AADT`), using the `k_factor` / `d_factor` from Section 4. This runs after the socioeconomic merge (Section 5) so that, column-wise, the socioeconomic fields sit between `d_factor` and `peak_hour_volume` in `combined_output.csv`.

In [9]:
master_df["peak_hour_volume"] = np.where(
    master_df["observation_unit"] == "AADT",
    master_df["observation_value"] * master_df["k_factor"] * master_df["d_factor"],
    np.nan,
)

master_df[[
    "functional_class", "through_lanes", "k_factor", "d_factor",
    "population", "avg annual pay (2025$)",
    "observation_value", "peak_hour_volume",
]].head(10)

,functional_class,through_lanes,k_factor,d_factor,population,avg annual pay (2025$),observation_value,peak_hour_volume
0,Principal Arterial - Other,2.0,NaN,NaN,12803.0,42714.26,3430,NaN
1,Principal Arterial - Other,2.0,NaN,NaN,13438.0,50038.36,3780,NaN
2,Principal Arterial - Other,2.0,NaN,NaN,14079.0,48037.89,3620,NaN
3,Principal Arterial - Other,2.0,NaN,NaN,14305.0,41403.34,3360,NaN
4,Principal Arterial - Other,2.0,NaN,NaN,14085.0,41811.95,3914,NaN
5,Principal Arterial - Other,2.0,NaN,NaN,13884.0,43117.64,4494,NaN
6,Principal Arterial - Other,2.0,NaN,NaN,14037.0,42038.45,5307,NaN
7,Principal Arterial - Other,4.0,NaN,NaN,14219.0,42009.30,5147,NaN
8,Principal Arterial - Other,4.0,NaN,NaN,14331.0,41013.07,4580,NaN
9,Principal Arterial - Other,4.0,NaN,NaN,14512.0,41639.00,5342,NaN


## 7. Post-Construction Capacity Expansion Equation

Replaces the old flat `Cap_base x N x f_hv x PHF x f_p x f_g` lookup with the full model from
*Proposed Post-Capacity Expansion Equation (rev2)*:

$$\text{PostCapacity}_i = C_{eng,i} \times \exp\big(\beta_{LM}\Delta LM_i + \beta_L \Delta L_i + \beta_{LT}\Delta LT_i + \beta_{RT}\Delta RT_i + \beta_W W_i + \beta_E E_i + \beta_S S_i + \beta_F F_i + \beta_A \Delta\ln(AADT_i) + \beta_P \Delta\ln(Pop_r) + \beta_Y \Delta\ln(Inc_r) + \beta_{SL} SL_i + \beta_{SIG} SIG_i + \beta_{AT} AT_i\big)$$

**$C_{eng}$ (engineering baseline)** is computed with the SPR20-597 HERS free-flow-speed method (Section 3
of the doc): FFS → lane-width penalty → signal adjustment → stop-sign adjustment → speed at capacity
(90% of FFS, per HCM density = 35 pc/mi/ln) → flow per lane → x lanes x heavy-vehicle factor. It uses each
POI's **post-construction** geometry, per the equation's definition of $C_{eng}$.

In [10]:
# Load calibration betas (and HCM scalars) from Cap_Cal_Values.csv.

cap_cal = pd.read_csv(CAP_CAL_PATH, encoding="cp1252")

PARAM_TO_KEY = {
    "Heavy Vehicle Factor": "f_hv",
    "Peak Hour Factor": "PHF",
    "Driver Population Factor": "f_p",
    "Grade Adjustment Factor": "f_g",
    "Lanemiles Elasticity": "BLM",
    "Lane Count Effect": "BL",
    "Left Turn Lane Effect": "BLT",
    "Right Turn Lane Effect": "BRT",
    "Lane Width Effect": "BW",
    "Managed Lane Effect": "BE",
    "Shoulder Widening Effect": "BS",
    "Functional Class Effect": "BF",
    "Speed Limit Effect": "BSL",
    "Signals per Mile Effect": "BSIG",
    "Area Type Effect": "BAT",
    "Local Demand Change": "BA",
    "Population Growth": "BP",
    "Income Growth": "BY",
}
cap_cal["_key"] = cap_cal["Parameter"].str.strip().map(PARAM_TO_KEY)
missing = cap_cal[cap_cal["_key"].isna()]
if len(missing):
    print("WARNING: unrecognized rows in Cap_Cal_Values.csv (not used):", missing["Parameter"].tolist())

cap_cal["_value"] = pd.to_numeric(cap_cal["Default Value"], errors="coerce")

bad_parse = cap_cal[cap_cal["_value"].isna()]
if len(bad_parse):
    print("WARNING: could not parse Default Value for:", bad_parse["Parameter"].tolist())

betas = cap_cal.dropna(subset=["_key"]).set_index("_key")["_value"].to_dict()

# Sanity check against the doc's Section 5 priors
for sym in ["BSIG", "BAT"]:
    if sym in betas and betas[sym] > 0:
        print(f"WARNING: {sym} = {betas[sym]} parsed as positive but the doc expects it negative "
              f"(see Proposed_Post-Capacity_Expansion_Equation_rev2, Section 5) - check Cap_Cal_Values.csv.")

f_hv = betas["f_hv"]  # heavy-vehicle factor, used in the Ceng calc below

betas

{'f_hv': 0.95,
 'PHF': 0.92,
 'f_p': 0.95,
 'f_g': 1.0,
 'BLM': 0.45,
 'BL': 0.35,
 'BLT': 0.1,
 'BRT': 0.06,
 'BW': 0.05,
 'BE': 0.15,
 'BS': 0.05,
 'BF': 0.1,
 'BSL': 0.05,
 'BSIG': -0.1,
 'BAT': -0.1,
 'BA': 1.0,
 'BP': 0.25,
 'BY': 0.25}

In [11]:
# Per-POI before/after geometry, used both for Ceng (post-construction geometry)
# and for the Delta/indicator variables in the exponential multiplier.
GEOM_COLS = [
    "through_lanes", "lane_width", "left_turn_lanes", "right_turn_lanes",
    "shoulder_width", "express_hov_hot_lanes", "speed_limit_mph",
    "signals_per_mile", "stop_signs_per_mile", "segment_length_miles",
]
for c in GEOM_COLS:
    master_df[c] = pd.to_numeric(master_df[c], errors="coerce")

geom_by_period = (
    master_df.groupby(["poi_id", "before_or_after"])[GEOM_COLS].mean().unstack("before_or_after")
)

def pre(col):
    return geom_by_period[col]["before"] if "before" in geom_by_period[col] else np.nan

def post(col):
    return geom_by_period[col]["after"] if "after" in geom_by_period[col] else np.nan

# post-construction functional class / area type, used for Ceng flow-type and AT level
_after_rows = master_df[master_df["before_or_after"] == "after"]
fc_post = _after_rows.groupby("poi_id")["functional_class"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else None)
at_post = _after_rows.groupby("poi_id")["area_type"].agg(lambda s: s.mode().iloc[0] if len(s.mode()) else None)

poi = pd.DataFrame(index=geom_by_period.index)

# --- geometry deltas (post - pre / pre) ---
lane_miles_pre = pre("through_lanes") * pre("segment_length_miles")
lane_miles_post = post("through_lanes") * post("segment_length_miles")
poi["d_lane_miles"] = (lane_miles_post - lane_miles_pre) / lane_miles_pre
poi["d_through_lanes"] = post("through_lanes") - pre("through_lanes")
poi["d_left_turn_lanes"] = post("left_turn_lanes") - pre("left_turn_lanes")
poi["d_right_turn_lanes"] = post("right_turn_lanes") - pre("right_turn_lanes")

# --- 0/1 indicators (W, E, S, SL) ---
poi["lane_width_improved"] = (post("lane_width") > pre("lane_width")).astype("Int64")
poi["express_lane_added"] = (post("express_hov_hot_lanes") > pre("express_hov_hot_lanes")).astype("Int64")
poi["shoulder_widened"] = (post("shoulder_width") > pre("shoulder_width")).astype("Int64")
poi["speed_limit_increased"] = (post("speed_limit_mph") > pre("speed_limit_mph")).astype("Int64")

# --- functional class improvement (F): no ranking data, so left at 0 unless enriched later ---
poi["functional_class_improved"] = 0

# --- levels used directly (SIG, AT), from post-construction state ---
poi["signals_per_mile_post"] = post("signals_per_mile")
poi["stop_signs_per_mile_post"] = post("stop_signs_per_mile")
poi["through_lanes_post"] = post("through_lanes")
poi["lane_width_post"] = post("lane_width")
poi["speed_limit_post"] = post("speed_limit_mph")
poi["functional_class_post"] = fc_post
poi["area_type_post"] = at_post
poi["area_type_level"] = at_post.astype(str).str.strip().str.lower().isin(["urban", "suburban"]).astype(int)

# --- demand delta: mean AADT pre vs post per POI ---
master_df["observation_value"] = pd.to_numeric(master_df["observation_value"], errors="coerce")
aadt_obs = master_df[master_df["observation_unit"] == "AADT"]
aadt_by_period = aadt_obs.groupby(["poi_id", "before_or_after"])["observation_value"].mean().unstack("before_or_after")
poi["d_ln_aadt"] = np.log(aadt_by_period["after"] / aadt_by_period["before"])

# --- regional population & income growth: pre vs post, straight from the county
# socioeconomic merge (Section 5) for each POI's own state/county/year ---
SOCIO_GROWTH_COLS = ["population", "avg annual pay (2025$)"]
socio_by_period = (
    master_df.groupby(["poi_id", "before_or_after"])[SOCIO_GROWTH_COLS].mean().unstack("before_or_after")
)

def socio_pre(col):
    return socio_by_period[col]["before"] if "before" in socio_by_period[col] else np.nan

def socio_post(col):
    return socio_by_period[col]["after"] if "after" in socio_by_period[col] else np.nan

poi["d_ln_pop"] = np.log(socio_post("population") / socio_pre("population"))
poi["d_ln_income"] = np.log(socio_post("avg annual pay (2025$)") / socio_pre("avg annual pay (2025$)"))

poi.head(10)

,d_lane_miles,d_through_lanes,d_left_turn_lanes,d_right_turn_lanes,lane_width_improved,express_lane_added,shoulder_widened,speed_limit_increased,functional_class_improved,signals_per_mile_post,stop_signs_per_mile_post,through_lanes_post,lane_width_post,speed_limit_post,functional_class_post,area_type_post,area_type_level,d_ln_aadt,d_ln_pop,d_ln_income
poi_id,,,,,,,,,,,,,,,,,,,,
6752,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,unknown,Rural Area,0.0,0.241733,0.076421,0.132821
6757,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,unknown,Rural Area,0.0,-0.274552,0.076421,0.132821
6763,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,unknown,Rural Area,0.0,-0.070415,0.071175,0.116088
6774,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,unknown,Rural Area,0.0,-0.078380,0.074956,0.088914
6788,0.0,0.0,0.0,0.0,0,0,0,0,0,1.0,0.0,4.0,12.0,55.0,freeway,rural,0.0,0.009838,0.037465,0.015270
6789,0.0,0.0,0.0,0.0,0,0,0,0,0,0.0,0.0,2.0,12.0,55.0,principal arterial,rural,0.0,-0.074592,0.009106,-0.011574
9834,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,principal arterial,rural,0.0,NaN,NaN,NaN
9836,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,principal arterial,rural,0.0,NaN,NaN,NaN
9860,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,principal arterial,rural,0.0,NaN,NaN,NaN


In [12]:
# SPR20-597 HERS free-flow-speed method -> engineering baseline capacity (Ceng), Sections 2-3 of the doc.
# Uses each POI's POST-construction geometry, since Ceng is defined for "the post-construction geometry."
UNINTERRUPTED_CLASSES = {"freeway", "ramp"}

fc_norm = poi["functional_class_post"].astype(str).str.strip().str.lower()
is_uninterrupted = fc_norm.isin(UNINTERRUPTED_CLASSES)
sigma = np.where(is_uninterrupted, 0.1427, 0.3907)
beta_hers = np.where(is_uninterrupted, 0.2092, 0.18378)

# No curvature/pavement-roughness data available -> assume straight road, good pavement
# (VCURVE = VROUGH = VSPLIM), per the doc's stated default for this case.
VSPLIM = poi["speed_limit_post"].astype(float)
VCURVE = VSPLIM
VROUGH = VSPLIM

base_ffs = (np.exp(sigma**2 / 2) * (VCURVE**(-1/beta_hers) + VROUGH**(-1/beta_hers) + VSPLIM**(-1/beta_hers)))**(-beta_hers)

# Lane-width penalty (12ft=0, 11ft=1.5mph, 10ft=4mph, <10ft=6.5mph)
lane_width = poi["lane_width_post"].astype(float)
fw = np.select([lane_width >= 12, lane_width >= 11, lane_width >= 10], [0, 1.5, 4], default=6.5)
ffs_lw = base_ffs - fw

# Signal adjustment (doc formula, NOT the buggy xlsx one - see markdown above)
nsig = poi["signals_per_mile_post"].astype(float)
zvdsig = np.where(nsig == 0, 0, 0.0687 * (1 - np.exp(-nsig / 24.4)))
ffs_sig = 1 / (1/ffs_lw + zvdsig)

# Stop-sign adjustment
nstp = poi["stop_signs_per_mile_post"].astype(float)
zvdstp = np.where(nstp == 0, 0, nstp * (1.9 + 0.067 * ffs_sig))
ffs_final = 1 / (1/ffs_sig + zvdstp / 1000)

# Speed at capacity (HCM density = 35 pc/mi/ln) -> flow per lane -> Ceng
v_cap = ffs_final * 0.9
s_cap = v_cap * 35
poi["engineering_capacity_ceng"] = s_cap * poi["through_lanes_post"].astype(float) * f_hv

poi[["functional_class_post", "speed_limit_post", "through_lanes_post", "engineering_capacity_ceng"]].head(10)

,functional_class_post,speed_limit_post,through_lanes_post,engineering_capacity_ceng
poi_id,,,,
6752,unknown,NaN,NaN,NaN
6757,unknown,NaN,NaN,NaN
6763,unknown,NaN,NaN,NaN
6774,unknown,NaN,NaN,NaN
6788,freeway,55.0,4.0,4659.911495
6789,principal arterial,55.0,2.0,2652.465768
9834,principal arterial,NaN,NaN,NaN
9836,principal arterial,NaN,NaN,NaN
9860,principal arterial,NaN,NaN,NaN


In [13]:
# Exponent X = sum(beta x variable), multiplier M = e^X, PostCapacity = Ceng x M
X = (
    betas["BLM"] * poi["d_lane_miles"]
    + betas["BL"] * poi["d_through_lanes"]
    + betas["BLT"] * poi["d_left_turn_lanes"]
    + betas["BRT"] * poi["d_right_turn_lanes"]
    + betas["BW"] * poi["lane_width_improved"]
    + betas["BE"] * poi["express_lane_added"]
    + betas["BS"] * poi["shoulder_widened"]
    + betas["BF"] * poi["functional_class_improved"]
    + betas["BA"] * poi["d_ln_aadt"]
    + betas["BP"] * poi["d_ln_pop"]  # regional population growth, from us_county_socioeconomic_2000_2025.csv
    + betas["BY"] * poi["d_ln_income"]  # regional income growth, from us_county_socioeconomic_2000_2025.csv
    + betas["BSL"] * poi["speed_limit_increased"]
    + betas["BSIG"] * poi["signals_per_mile_post"]
    + betas["BAT"] * poi["area_type_level"]
)

poi["capacity_multiplier"] = np.exp(X)
poi["peak_hour_capacity"] = poi["engineering_capacity_ceng"] * poi["capacity_multiplier"]

# Merge the POI-level results back onto every observation row for that POI
_poi_out = poi[["engineering_capacity_ceng", "capacity_multiplier", "peak_hour_capacity"]].reset_index()
master_df = master_df.drop(columns=["engineering_capacity_ceng", "capacity_multiplier", "peak_hour_capacity"], errors="ignore")
master_df = master_df.merge(_poi_out, on="poi_id", how="left")

master_df[[
    "poi_id", "functional_class", "before_or_after",
    "engineering_capacity_ceng", "capacity_multiplier", "peak_hour_capacity",
]].head(10)

,poi_id,functional_class,before_or_after,engineering_capacity_ceng,capacity_multiplier,peak_hour_capacity
0,111542,Principal Arterial - Other,before,NaN,<NA>,<NA>
1,111542,Principal Arterial - Other,before,NaN,<NA>,<NA>
2,111542,Principal Arterial - Other,before,NaN,<NA>,<NA>
3,111542,Principal Arterial - Other,before,NaN,<NA>,<NA>
4,111542,Principal Arterial - Other,before,NaN,<NA>,<NA>
5,111542,Principal Arterial - Other,before,NaN,<NA>,<NA>
6,111542,Principal Arterial - Other,before,NaN,<NA>,<NA>
7,111542,Principal Arterial - Other,after,NaN,<NA>,<NA>
8,111542,Principal Arterial - Other,after,NaN,<NA>,<NA>
9,111542,Principal Arterial - Other,after,NaN,<NA>,<NA>


## 8. Spot-check the merged data

Displays a narrowed view of just the columns most relevant for spot-checking, the join keys, dates, before/after labels, and a few geometric attributes (lane counts, shoulder width), to visually verify the merges worked correctly before writing anything out.

In [14]:
master_df[[
    "project_id", "poi_id", "observation_date", "date_open_actual",
    "before_or_after", "before_or_after_binary", "years_from_open",
    "through_lanes", "right_turn_lanes", "shoulder_width",
    "functional_class", "k_factor", "d_factor",
    "population", "avg annual pay (2025$)",
    "peak_hour_volume", "peak_hour_capacity",
]]

,project_id,poi_id,observation_date,date_open_actual,before_or_after,before_or_after_binary,years_from_open,through_lanes,right_turn_lanes,shoulder_width,functional_class,k_factor,d_factor,population,avg annual pay (2025$),peak_hour_volume,peak_hour_capacity
0,1-180.10,111542,2001-01-01,NaT,before,0,-19.0,2.0,1.0,NaN,Principal Arterial - Other,NaN,NaN,12803.0,42714.26,NaN,<NA>
1,1-180.10,111542,2004-01-01,NaT,before,0,-16.0,2.0,1.0,NaN,Principal Arterial - Other,NaN,NaN,13438.0,50038.36,NaN,<NA>
2,1-180.10,111542,2007-01-01,NaT,before,0,-13.0,2.0,1.0,NaN,Principal Arterial - Other,NaN,NaN,14079.0,48037.89,NaN,<NA>
3,1-180.10,111542,2010-01-01,NaT,before,0,-10.0,2.0,1.0,NaN,Principal Arterial - Other,NaN,NaN,14305.0,41403.34,NaN,<NA>
4,1-180.10,111542,2013-01-01,NaT,before,0,-7.0,2.0,1.0,NaN,Principal Arterial - Other,NaN,NaN,14085.0,41811.95,NaN,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4836,TA-M352,10707,2021-01-01,NaT,after,1,31.0,NaN,NaN,NaN,principal arterial,0.12,0.6,442829.0,76136.66,3485.376,<NA>
4837,TA-M352,10707,2022-01-01,NaT,after,1,32.0,NaN,NaN,NaN,principal arterial,0.12,0.6,444023.0,74580.81,3414.024,<NA>
4838,TA-M352,10707,2023-01-01,NaT,after,1,33.0,NaN,NaN,NaN,principal arterial,0.12,0.6,448612.0,73613.20,3673.944,<NA>
4839,TA-M352,10707,2024-01-01,NaT,after,1,34.0,NaN,NaN,NaN,principal arterial,0.12,0.6,453755.0,74707.35,4224.744,<NA>


## 9. Write out the combined CSV

Adds a blank `comment` column (for manual notes) as the very last column, then saves `master_df` to `combined_output.csv` inside `DATA_ROOT` (the same folder your project subfolders live in, alongside `validation_report.txt`) and confirms how many rows/columns were written. This file is the input for the quantile regression model-fitting notebook.

In [15]:
output_df = master_df.drop(columns=["engineering_capacity_ceng", "capacity_multiplier"], errors="ignore")
output_df["comment"] = ""

output_path = os.path.join(DATA_ROOT, "combined_output.csv")
output_df.to_csv(output_path, index=False)
print(f"Wrote {len(output_df)} rows and {len(output_df.columns)} columns to {output_path}")

Wrote 4841 rows and 49 columns to C:\Users\User\Documents\Code\forecastcards_ba_data\data\combined_output.csv
